# 11 Sample Purification

本实验对 `train / val / test` 三个数据集都执行样本提纯。其中：
- `train`：执行规则提纯 + 分层抽样
- `val / test`：执行规则提纯，不做分层抽样，尽量保持评估分布自然


In [ ]:
from __future__ import annotations

from pathlib import Path
import sys

import pandas as pd
from IPython.display import Markdown, display

PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "src").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

for candidate in (PROJECT_ROOT, PROJECT_ROOT / "src"):
    candidate_text = str(candidate)
    if candidate_text not in sys.path:
        sys.path.insert(0, candidate_text)

from src.features.feature_selector import CreditFeatureSelector
from src.features.preprocessor import CreditDataPreprocessor
from src.features.sample_purifier import CreditSamplePurifier
from src.models.model_evaluator import CreditModelEvaluator
from src.models.risk_classifier import CreditRiskClassifier

DATA_DIR = PROJECT_ROOT / "data" / "processed"
DOCS_DIR = PROJECT_ROOT / "docs"
DOCS_DIR.mkdir(parents=True, exist_ok=True)
CLASS_NAMES = ["正常类", "关注类", "次级/可疑类", "损失类"]


In [ ]:
train_df = pd.read_csv(DATA_DIR / "train.csv", low_memory=False)
val_df = pd.read_csv(DATA_DIR / "val.csv", low_memory=False)
test_df = pd.read_csv(DATA_DIR / "test.csv", low_memory=False)

purifier = CreditSamplePurifier()


In [ ]:
def purify_split(df: pd.DataFrame, name: str, apply_sampling: bool) -> tuple[pd.DataFrame, dict, dict]:
    before_distribution = purifier.get_sample_distribution(df, name=f"{name}提纯前")
    filtered_df = purifier.filter_invalid_normal_samples(df)
    filtered_distribution = purifier.get_sample_distribution(filtered_df, name=f"{name}规则提纯后")
    final_df = (
        purifier.stratified_sampling_normal_samples(filtered_df, target_ratio=4)
        if apply_sampling
        else filtered_df.copy()
    )
    after_distribution = purifier.get_sample_distribution(final_df, name=f"{name}最终")
    return final_df, before_distribution, {
        "filtered": filtered_distribution,
        "final": after_distribution,
    }

purified_train_df, train_before, train_after = purify_split(train_df, "train", apply_sampling=True)
purified_val_df, val_before, val_after = purify_split(val_df, "val", apply_sampling=False)
purified_test_df, test_before, test_after = purify_split(test_df, "test", apply_sampling=False)

purifier.plot_distribution_comparison(train_df, purified_train_df)

purified_train_path = DATA_DIR / "purified_train.csv"
purified_val_path = DATA_DIR / "purified_val.csv"
purified_test_path = DATA_DIR / "purified_test.csv"

purified_train_df.to_csv(purified_train_path, index=False)
purified_val_df.to_csv(purified_val_path, index=False)
purified_test_df.to_csv(purified_test_path, index=False)

display(Markdown("已输出提纯后的 `train / val / test` 三个数据集。"))


In [ ]:
def train_and_evaluate(train_frame: pd.DataFrame, test_frame: pd.DataFrame) -> dict:
    y_train = train_frame["preloan_risk_label"].astype(int)
    y_test = test_frame["preloan_risk_label"].astype(int)

    preprocessor = CreditDataPreprocessor(target_column="preloan_risk_label")
    X_train_processed = preprocessor.fit_transform(train_frame)
    X_test_processed = preprocessor.transform(test_frame)

    selector = CreditFeatureSelector(top_k_features=80, use_pca=False, n_estimators=100)
    X_train_selected = selector.fit_transform(X_train_processed, y_train)
    X_test_selected = selector.transform(X_test_processed)

    model = CreditRiskClassifier(model_type="lightgbm", class_weight_mode="none")
    model.fit(X_train_selected, y_train)

    evaluator = CreditModelEvaluator(
        model=model,
        X_test=X_test_selected,
        y_test=y_test,
        class_names=CLASS_NAMES,
    )
    metrics = evaluator.evaluate_imbalanced_multiclass()
    return {
        "accuracy": float(metrics["accuracy"]),
        "macro_f1": float(metrics["macro_f1"]),
        "weighted_f1": float(metrics["weighted_f1"]),
        "ks_value": float(metrics["ks_value"]),
        "classification_report_named": metrics["classification_report_named"],
        "confusion_matrix": metrics["confusion_matrix"],
    }


In [ ]:
original_metrics = train_and_evaluate(train_df, test_df)
purified_metrics = train_and_evaluate(purified_train_df, purified_test_df)

comparison_df = pd.DataFrame([
    {
        "训练集方案": "原始 train -> 原始 test",
        "准确率": original_metrics["accuracy"],
        "Macro-F1": original_metrics["macro_f1"],
        "Weighted-F1": original_metrics["weighted_f1"],
        "KS": original_metrics["ks_value"],
    },
    {
        "训练集方案": "提纯 train -> 提纯 test",
        "准确率": purified_metrics["accuracy"],
        "Macro-F1": purified_metrics["macro_f1"],
        "Weighted-F1": purified_metrics["weighted_f1"],
        "KS": purified_metrics["ks_value"],
    },
])
comparison_df


In [ ]:
report_lines = [
    "# 样本提纯报告",
    "",
    "## 说明",
    "- 本报告对应当前正式 `4_class_merge_23` 口径。",
    "- 当前对 `train / val / test` 三个数据集均执行了规则提纯。",
    "- 其中仅 `train` 额外执行了分层抽样，`val / test` 保持提纯后自然分布。",
    "",
    "## 分布变化",
    f"- train 提纯前：`{train_before}`",
    f"- train 规则提纯后：`{train_after['filtered']}`",
    f"- train 最终：`{train_after['final']}`",
    f"- val 提纯前：`{val_before}`",
    f"- val 规则提纯后：`{val_after['filtered']}`",
    f"- val 最终：`{val_after['final']}`",
    f"- test 提纯前：`{test_before}`",
    f"- test 规则提纯后：`{test_after['filtered']}`",
    f"- test 最终：`{test_after['final']}`",
    "",
    "## 输出文件",
    "- `data/processed/purified_train.csv`",
    "- `data/processed/purified_val.csv`",
    "- `data/processed/purified_test.csv`",
    "- `results/figures/sample_purification_comparison.png`",
    "",
    "## 模型对比",
    comparison_df.to_markdown(index=False),
    "",
    "## 各类别指标摘要",
    "### 原始 train -> 原始 test",
    pd.DataFrame(original_metrics["classification_report_named"]).transpose().to_markdown(),
    "",
    "### 提纯 train -> 提纯 test",
    pd.DataFrame(purified_metrics["classification_report_named"]).transpose().to_markdown(),
]

report_path = DOCS_DIR / "sample_purification_report.md"
report_path.write_text("
".join(report_lines), encoding="utf-8")
display(Markdown(f"报告已保存到 `docs/{report_path.name}`"))
